# Exercise 2: Surface and lemma novelty

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Evaluation and Benchmarks · version 0.1.0*

> Partition tagging errors by surface and gold-lemma novelty, reporting counts and undefined empty cells.

**Setup.** `pip install "stanza==1.11.0" sacrebleu torch transformers sentencepiece llama-cpp-python fasttext-wheel pyarrow requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Exercises 1 and 2 tag the **UD 2.17** Kazakh-KTB and Turkish-IMST test files with Stanza 1.11.0 over the gold words (models from `STANZA_RESOURCES_DIR`, files cached in `UD217_DIR` and checked against their SHA-256 hashes). Exercise 3 uses the X-WMT English–Turkish test set (TIL release, commit `7d988cc`), TurkicNLP's Apertium analyser and generator for Turkish and NLLB-200 distilled 600M (`NLLB_PATH`). Exercises 4 and 5 use Qwen2.5-1.5B-Instruct as an 8-bit GGUF file run with llama.cpp on the CPU (`QWEN_GGUF`, its tokeniser from `QWEN_TOKENIZER`) and GlotLID v3 (`GLOTLID_PATH`). Exercise 4 needs the Kazakh TUMLU-mini files (`TUMLU_DIR`, revision `3523de7`), checked against the hashes of the book's audit. Model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

The training vocabulary is the official UD training split of each treebank (lower-cased, with Turkish dotted/dotless *i* handled). Each non-punctuation test word falls into one of four cells; UPOS, feature-bundle and lemma error rates are computed per cell.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {   # UD 2.17 release files
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Kazakh-KTB/kk_ktb-ud-train.conllu": "6297597914a3f319db573573f927a6577c702f5fa00dcd653162aab46026aeaa",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
}
UD_DIR = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def ud_path(key):
    path = UD_DIR / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    if sha256(path) != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos feats")
def read(key):
    """Sentences (lists of Word) of a UD file; multiword-token ranges and empty nodes are skipped."""
    sents, words = [], []
    for line in ud_path(key).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append(words)
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[5]))
    return sents

def feats(s):
    return dict(f.split("=", 1) for f in (s or "_").split("|") if "=" in f)

def tag(lang, key):
    """Stanza 1.11 POS/feature/lemma predictions for the gold words of a UD file (cached)."""
    gold = read(key)
    path = CACHE / f"ch15-stanza-{lang}-{SHA256[key][:12]}.json"
    if not path.exists():
        import logging, stanza
        logging.getLogger("stanza").setLevel(logging.ERROR)
        nlp = stanza.Pipeline(lang, processors="tokenize,pos,lemma", tokenize_pretokenized=True, verbose=False,
                              model_dir=os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser())),
                              download_method=None)
        doc = nlp([[w.form for w in s] for s in gold])
        pred = [[[w.lemma or "_", w.upos or "_", w.feats or "_"] for w in s.words] for s in doc.sentences]
        path.write_text(json.dumps(pred, ensure_ascii=False))
    pred = json.loads(path.read_text())
    assert [len(s) for s in pred] == [len(s) for s in gold]
    return gold, [[Word(g.id, g.form, *p) for g, p in zip(gs, ps)] for gs, ps in zip(gold, pred)]

def wilson(k, n, z=1.959963984540054):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

# ---- Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) and GlotLID, for Exercises 4 and 5 ---------------
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
_LLM = {}
def llm():
    if "m" not in _LLM:
        from llama_cpp import Llama
        _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
    return _LLM["m"]

REPLIES = CACHE / "ch15-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["value"]
def cached(key_obj, compute):
    key = hashlib.sha1(json.dumps(key_obj, ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        _CACHED[key] = compute()
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "input": key_obj, "value": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

def chat(messages, max_tokens=120):
    """Greedy reply (cached)."""
    return cached(["chat", messages, max_tokens], lambda: llm().create_chat_completion(
        messages=messages, temperature=0.0, max_tokens=max_tokens)["choices"][0]["message"]["content"].strip())

_LID = {}
def lid(text):
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

print("UD files:", ", ".join(k.split("/")[1] for k in SHA256))

In [ ]:
def low(s, lang):
    return s.replace("I", "ı").replace("İ", "i").lower() if lang == "tr" else s.lower()
for lang, train, test in (("tr", "UD_Turkish-IMST/tr_imst-ud-train.conllu", "UD_Turkish-IMST/tr_imst-ud-test.conllu"),
                          ("kk", "UD_Kazakh-KTB/kk_ktb-ud-train.conllu", "UD_Kazakh-KTB/kk_ktb-ud-test.conllu")):
    tr = read(train)
    forms = {low(w.form, lang) for s in tr for w in s}; lemmas = {low(w.lemma, lang) for s in tr for w in s}
    gold, pred = tag(lang, test)
    cells = collections.defaultdict(lambda: collections.Counter())
    for gs, ps in zip(gold, pred):
        for g, p in zip(gs, ps):
            if g.upos == "PUNCT":
                continue
            c = (("seen" if low(g.form, lang) in forms else "unseen") + " surface / " +
                 ("seen" if low(g.lemma, lang) in lemmas else "unseen") + " lemma")
            cells[c]["n"] += 1; cells[c]["upos"] += p.upos != g.upos
            cells[c]["feats"] += feats(p.feats) != feats(g.feats); cells[c]["lemma"] += low(p.lemma, lang) != low(g.lemma, lang)
    print(f"{test.split('/')[0][3:]}: training vocabulary {len(forms):,} surface forms and {len(lemmas):,} lemmas from {len(tr):,} sentences")
    print(f"  {'cell':<30}{'words':>7}{'UPOS err':>10}{'feats err':>10}{'lemma err':>10}")
    for c in ("seen surface / seen lemma", "unseen surface / seen lemma", "seen surface / unseen lemma", "unseen surface / unseen lemma"):
        x = cells.get(c)
        if not x:
            print(f"  {c:<30}{0:>7}{'undefined':>10}{'undefined':>10}{'undefined':>10}"); continue
        print(f"  {c:<30}{x['n']:>7,}" + "".join(f"{x[k] / x['n']:10.1%}" for k in ("upos", "feats", "lemma")))
    tot = sum(x["n"] for x in cells.values())
    print(f"  surface OOV {sum(x['n'] for c, x in cells.items() if c.startswith('unseen')) / tot:.1%}; "
          f"lemma OOV {sum(x['n'] for c, x in cells.items() if 'unseen lemma' in c) / tot:.1%} of {tot:,} non-punctuation words")

## Worked solution

Output of `experiments/ch15_exercises/run_ch15_exercises.py`:

```
Turkish-IMST: training vocabulary 12,345 surface forms and 4,684 lemmas from 3,435 sentences
  cell                            words  UPOS err feats err lemma err
  seen surface / seen lemma       5,395      4.8%      6.3%      1.1%
  unseen surface / seen lemma     1,874      5.7%     20.5%      8.2%
  seen surface / unseen lemma        12     58.3%     75.0%     83.3%
  unseen surface / unseen lemma     818     13.6%     17.0%     16.1%
  surface OOV 33.2%; lemma OOV 10.2% of 8,099 non-punctuation words
Kazakh-KTB: training vocabulary 319 surface forms and 252 lemmas from 31 sentences
  cell                            words  UPOS err feats err lemma err
  seen surface / seen lemma       1,043      2.8%      2.2%      0.7%
  unseen surface / seen lemma     1,226      1.6%      3.5%      1.5%
  seen surface / unseen lemma        18     11.1%      5.6%     22.2%
  unseen surface / unseen lemma   5,738      1.3%      2.1%      1.7%
  surface OOV 86.8%; lemma OOV 71.7% of 8,025 non-punctuation words
```

**Report each cell separately.** Turkish IMST has a 6.3% bundle error rate for seen forms with seen lemmas, 20.5% for unseen forms with seen lemmas, and 17.0% for unseen forms with unseen lemmas. The corresponding lemma error rates are 1.1%, 8.2% and 16.1%. Surface OOV is 33.2%, while lemma OOV is 10.2%. These differences locate errors but do not prove that inflectional novelty causes them; the cells differ in frequency, word classes and other properties.

The seen-form/unseen-lemma cell contains only twelve Turkish words. Its 75.0% bundle error rate should be read with that denominator, not merged silently with other seen forms.

For Kazakh, 86.8% of non-punctuation test words are absent from the official training vocabulary. This is not novelty relative to the model's training: Stanza used the test file. Its bundle error rates across the four cells are 2.2%, 3.5%, 5.6% and 2.1%. None of the cells is empty; an empty cell would have an undefined rate.

## Your turn — extension

1. Add a fifth cell for words whose lemma was seen only with a different UPOS.
2. Use the Turkish BOUN training split as the vocabulary and see how the cells move.